In [1]:
import pickle
import networkx as nx
import pennylane as qml
from joblib import Parallel, delayed
from pennylane import numpy as np
from itertools import combinations


In [47]:
P = np.array([
    [1/3, 0.0],
    [1/3, 0.25],
    [0.0,     0.25],
    [0.0,     0.25],
    [1/3, 0.25]
], requires_grad=True)

w = np.ones(2)

In [48]:
def make_M_V(P, w):
    M = P @ np.diag(w) @ P.T
    return M, np.diag(P @ w) - M

In [49]:
M,V = make_M_V(P,w)

In [50]:
print(V)

[[ 0.22222222 -0.11111111  0.          0.         -0.11111111]
 [-0.11111111  0.40972222 -0.0625     -0.0625     -0.17361111]
 [ 0.         -0.0625      0.1875     -0.0625     -0.0625    ]
 [ 0.         -0.0625     -0.0625      0.1875     -0.0625    ]
 [-0.11111111 -0.17361111 -0.0625     -0.0625      0.40972222]]


In [44]:
#Q is quadratioc objective rather than autocorrelation matrix
def max_cut_from_Q(Q, 
                   seed: int,
                   num_layers: int,
                   scaling_constant: float,
                   step_size: float, 
                   max_trials: int,
                   threshold: float,
                  ):
    rng = np.random.default_rng(seed=seed)
    n_wires = Q.shape[0]
    triu_indices = np.triu_indices(Q.shape[0])
    edge_list = []
    for (u, v) in zip(triu_indices[0], triu_indices[1]):
        if u == v:
            continue
        elif Q[u][v] != 0.0:
            edge_list.append((u, v))
    num_edges = len(edge_list)
    print("Number of edges: ", num_edges)
    beta_params = scaling_constant * rng.random([n_wires, num_layers], requires_grad=True)
    gamma_params = scaling_constant * rng.random([num_edges, num_layers], requires_grad=True)
    def U_B(beta):
        for wire in range(n_wires):
            qml.RX(2 * beta[wire], wires=wire)
    def U_C(gamma):
        edge_idx = 0
        for u, v in edge_list: 
            w = Q[u][v]
            qml.CNOT(wires=(u, v))
            qml.RZ(-w * gamma[edge_idx], wires=v)
            qml.CNOT(wires=(u, v))
            edge_idx += 1 
    dev = qml.device('default.qubit', wires=n_wires)
    @qml.qnode(dev)
    def circuit(gammas, betas):
        for wire in range(n_wires):
            qml.Hadamard(wires=wire)
        for k in range(num_layers):
            U_C(gammas[:, k])
            U_B(betas[:, k])
        return [qml.expval(qml.PauliZ(u) @ qml.PauliZ(v)) for u, v in edge_list]
    
    def objective(gammas, betas):
        expectation_values = circuit(gammas, betas)
        expectation_values = np.array(expectation_values)
        values = 0.0
        for u in range(n_wires):
            values += Q[u][u]
        edge_idx = 0
        for u, v in edge_list:
            w  = Q[u][v]
            values += 2 * expectation_values[edge_idx] * w
            edge_idx += 1
        return -values

    opt = qml.AdamOptimizer(stepsize=step_size)
    objective_list = [objective(gamma_params, beta_params)]
    params_list = [(gamma_params, beta_params)]
    for i in range(max_trials):
        (gamma_params, beta_params), current_obj = opt.step_and_cost(objective, gamma_params, beta_params)
        if (i + 1) % 50 == 0:
            print("Current obj: ", current_obj)
            if np.abs(current_obj - objective_list[-1]) <= threshold:
                objective_list.append(current_obj)
                params_list.append((gamma_params, beta_params))
                break
        objective_list.append(current_obj)
        params_list.append((gamma_params, beta_params))
    result = [objective_list, params_list]
    best_params = result[1][np.argmin(result[0])]
    best_qaoa_objective = np.min(result[0])
    
    #print("Best param: ", best_params)
    print("Best QAOA obj: ", best_qaoa_objective)
    
    dev_sampling = qml.device('default.qubit', wires=n_wires)
    @qml.set_shots(shots = 10000)
    @qml.qnode(dev_sampling)
    def sampling_circuit(gammas, betas):
        for wire in range(n_wires):
            qml.Hadamard(wires=wire)
        for k in range(num_layers):
            U_C(gammas[:, k])
            U_B(betas[:, k])
        return qml.sample()
    best_gammas, best_betas = best_params
    samples = sampling_circuit(best_gammas, best_betas)
    values = []
    formatted_samples = []
    for sample in samples:
        new_x = 2*np.array(sample)-1
        value = new_x@Q@new_x.transpose()
        formatted_samples.append(sample)
        values.append(value)
    print("Best objective:", np.max(values))
    print("Average objective:", np.mean(values))
    print("Best x:", formatted_samples[np.argmax(values)])
    return [np.mean(values), best_qaoa_objective, best_params] #np.max(values), formatted_samples[np.argmax(values)], 





In [ ]:
def run_graph(idx, num_layers, run_type, graph_type):
    if idx < 10:
        hypergraph_name = f"0{idx}"
    else:
        hypergraph_name = f"{idx}"
    print("Name: ", hypergraph_name)
    P = np.load(f'data/{graph_type}/P/{hypergraph_name}.npy')
    w = np.load(f'data/{graph_type}/w/{hypergraph_name}.npy')
    M, V = make_M_V(P, w)
    Vlist = []
    Q = V
    Vlist = [w[i] * (np.diag(P[:, i]) - P[:, i].reshape(P.shape[0], 1) * P[:, i].reshape(1, P.shape[0])) for i in range(P.shape[1])]
    if run_type == "max":
        step_size = 5e-2
    elif run_type == "fair":
        step_size = 5e-2
    threshold = 1e-8
    max_trials = 300
    scaling_constant = 5e-2
    seed = 42
    if run_type == "max":
        results  =  max_cut_from_Q(Q=Q,
                                    seed = seed,
                                    num_layers = num_layers,
                                    scaling_constant = scaling_constant,
                                    step_size = step_size, 
                                    max_trials = max_trials,
                                    threshold = threshold)
        print("Best QAOA obj: ", results[1])
        print("Best average max:", results[0])
    elif run_type == "fair":
        results  =  fair_cut_from_Q(Vlist = Vlist, #
                                    seed = seed,
                                    num_layers = num_layers,
                                    scaling_constant = scaling_constant,
                                    step_size = step_size, 
                                    max_trials = max_trials,
                                    threshold = threshold)
        print("Best QAOA obj: ", results[1])
        print("Best fair sampling:", results[0])
    return results
